 # Section 1: Setup, Unified Paths & Global Settings

 This cell loads essential utilities, configures global Pandas display parameters,

 silences silent-downcasting warnings, and dynamically maps project directories.

In [1]:
# 3. Path Selection Toggle

# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR
USE_LOCAL_DATA = True  # Set False for GLOBAL_DATA_DIR

In [2]:
%load_ext autoreload
%autoreload 2

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from core.contracts import EngineInput, FilterPack
from core.paths import (
    GLOBAL_DATA_DIR,
    GLOBAL_PROCESSED_DIR,
    LOCAL_DATA_DIR,
    NOTEBOOKS_RLVR_ROOT,
    OUTPUT_DIR,
    PROJECT_ROOT,
)
from core.settings import TradingConfig
from core.utils import SystemUtils as SU
from data_pipeline import generate_features
from walk_forward import (
    AlphaEngine,
    create_walk_forward_analyzer,
    run_headless_simulation,
)

# 1. Global Configuration Options
config = TradingConfig()

# 2. Display Settings & Silence Warnings
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 3000)

# 3. Path Selection Toggle
# USE_LOCAL_DATA = False  # Set False for GLOBAL_DATA_DIR

data_dir = LOCAL_DATA_DIR if USE_LOCAL_DATA else GLOBAL_DATA_DIR
processed_dir = GLOBAL_PROCESSED_DIR
output_dir = OUTPUT_DIR

# 4. Define Target File Paths
ohlcv_filename = (
    "df_ohlcv.parquet" if USE_LOCAL_DATA else "df_OHLCV_stocks_etfs.parquet"
)
fed_filename = (
    "df_fed.parquet"
    if USE_LOCAL_DATA
    else "High_Yield_Spread_T10Y2Y_Spread.parquet"
)

raw_paths = {
    "df_ohlcv": data_dir / ohlcv_filename,
    "df_fed": data_dir / fed_filename,
    "df_indices": data_dir / "df_indices.parquet",
}

features_df_path = data_dir / "features_df.parquet"
macro_df_path = data_dir / "macro_df.parquet"

print(
    f"Paths Resolved:\n"
    f" - Active Mode:          {'LOCAL' if USE_LOCAL_DATA else 'GLOBAL'}\n"
    f" - Active Data Dir:       {data_dir}\n"
    f" - Global Processed Dir: {GLOBAL_PROCESSED_DIR}\n"
    f" - Output Dir:           {OUTPUT_DIR}\n"
)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

Paths Resolved:
 - Active Mode:          LOCAL
 - Active Data Dir:       C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
 - Global Processed Dir: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
 - Output Dir:           C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output



# Section 2: Ingest Datasets & Generate Features

Loads raw input files and retrieves pre-calculated features or runs feature generation.

In [3]:
# Load Raw Files
raw_dfs = {}

for name, path in raw_paths.items():
    if path.exists():
        raw_dfs[name] = pd.read_parquet(path)
        print(f"[OK] Successfully loaded {name:<10} from {path.name}")
    else:
        print(f"[ERROR] File not found: {path}")

df_ohlcv = raw_dfs.get("df_ohlcv")
df_fed = raw_dfs.get("df_fed")
df_indices = raw_dfs.get("df_indices")

[OK] Successfully loaded df_ohlcv   from df_ohlcv.parquet
[OK] Successfully loaded df_fed     from df_fed.parquet
[OK] Successfully loaded df_indices from df_indices.parquet


In [4]:
# Load Processed Features or Generate If Missing / Running Global
if USE_LOCAL_DATA and features_df_path.exists() and macro_df_path.exists():
    features_df = pd.read_parquet(features_df_path)
    macro_df = pd.read_parquet(macro_df_path)
    print(f"[OK] Loaded features_df from local disk | Shape: {features_df.shape}")
    print(f"[OK] Loaded macro_df from local disk    | Shape: {macro_df.shape}")
else:
    reason = (
        "USE_LOCAL_DATA is False"
        if not USE_LOCAL_DATA
        else "Local feature files missing"
    )
    print(f"[INFO] {reason}. Generating features (~6 mins)...")

    features_df, macro_df = generate_features(
        df_ohlcv=df_ohlcv, df_indices=df_indices, df_fed=df_fed, config=config
    )
    print(f"[OK] Generated features_df  | Shape: {features_df.shape}")
    print(f"[OK] Generated macro_df     | Shape: {macro_df.shape}")

[OK] Loaded features_df from local disk | Shape: (8568638, 21)
[OK] Loaded macro_df from local disk    | Shape: (7477, 10)


In [5]:
# Cache Processed DataFrames to GLOBAL_PROCESSED_DIR when running Global
if not USE_LOCAL_DATA:
    GLOBAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving DataFrames to: {GLOBAL_PROCESSED_DIR}\n")

    dfs_to_save = {
        "df_ohlcv.parquet": df_ohlcv,
        "df_indices.parquet": df_indices,
        "df_fed.parquet": df_fed,
        "features_df.parquet": features_df,
        "macro_df.parquet": macro_df,
    }

    for filename, df in dfs_to_save.items():
        if df is not None:
            save_path = GLOBAL_PROCESSED_DIR / filename
            df.to_parquet(save_path)
            print(f"[OK] Saved {filename:<20} | Shape: {df.shape}")
        else:
            print(f"[SKIP] {filename:<19} | DataFrame missing in memory")

In [6]:
###############################
###############################

In [7]:
print("🚀 Unstacking Wide Matrices...")

# 1. Unstack directly.
# (Since df_ohlcv is already a perfect grid of master_dates x tickers,
# unstacking automatically creates perfectly aligned wide dataframes).
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)
df_atrp_wide = features_df["ATRP"].unstack(level=0)
df_trp_wide = features_df["TRP"].unstack(level=0)

print("Applying terminal NaN replacements for model inputs...")

# 2. Terminal Fills ONLY.
# Note: 0-to-NaN conversion and bounded ffills were ALREADY applied upstream to df_ohlcv.
# Applying ffill here would incorrectly double your max_data_gap_ffill limit!
df_close_wide = df_close_wide.fillna(config.nan_price_replacement)
df_atrp_wide = df_atrp_wide.fillna(0.0)
df_trp_wide = df_trp_wide.fillna(0.0)

print("All wide dataframes unstacked and prepared for downstream modeling.")
print(
    f"[OK] Pipeline Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)

🚀 Unstacking Wide Matrices...
Applying terminal NaN replacements for model inputs...
All wide dataframes unstacked and prepared for downstream modeling.
[OK] Pipeline Complete. Tickers: 1709, Days: 7477


 # Section 3: In-Memory Structural Previews

 Outputs head and tail previews of the core dataframes.

In [8]:
# print("\n" + "=" * 50)
# print("DATAFRAME STRUCTURAL PREVIEWS (HEAD & TAIL)")
# print("=" * 50)

# # 1. Preview wide matrix prices
# if "df_close_wide" in locals():
#     print("\n=== df_close_wide (Prices) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(df_close_wide.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(df_close_wide.tail(3))

# # 2. Preview multi-index features
# if "features_df" in locals():
#     print("\n=== features_df (Multi-Index Features) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(features_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(features_df.tail(3))

# # 3. Preview macro data
# if "macro_df" in locals():
#     print("\n=== macro_df (Macro Data) ===")
#     print("--- HEAD (First 3 Rows) ---")
#     display(macro_df.head(3))
#     print("\n--- TAIL (Last 3 Rows) ---")
#     display(macro_df.tail(3))

 # Section 4: Engine Initialization & Stage 1 Analysis

In [9]:
# Instantiate Master Engine
engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)
print("Master AlphaEngine Ready.")

Master AlphaEngine Ready.


In [10]:
# import pandas as pd

# target_date = pd.Timestamp("2025-09-01")

# # Use level=1 for the date level (since level names aren't 'timestamp')
# dates_in_index = df_ohlcv.index.get_level_values(1).normalize().unique()
# print(f"Labor Day 2025-09-01 in index: {target_date in dates_in_index}")

# # Check if date exists (one-liner)
# any_match = (df_ohlcv.index.get_level_values(1).normalize() == target_date).any()
# print(f"Any match: {any_match}")

# # If you want to see which tickers have data on that date:
# if any_match:
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     tickers_on_date = df_ohlcv.index.get_level_values(0)[mask].unique()
#     print(f"Tickers on 2025-09-01: {tickers_on_date}")
#     print(f"Count: {len(tickers_on_date)}")

# # Extract all data for that specific date
# if any_match:
#     # Using boolean mask (works regardless of index sorting)
#     mask = df_ohlcv.index.get_level_values(1).normalize() == target_date
#     df_target = df_ohlcv[mask]
#     print(df_target.head())

In [11]:
# Configuration for Interactive UI (Stage 1 Setup)
_inputs = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp("2026-08-05"),
    lookback_period=189,
    holding_period=config.holding_period,
    metric="Sharpe (TRP)",
    benchmark_ticker=config.benchmark_ticker,
    rank_start=1,
    rank_end=50,  # Plot only display 50 tickers
    debug=False,
)

analyzer1, stage1_pack = create_walk_forward_analyzer(
    engine, _inputs, universe_subset=None
)
analyzer1.show()

In [12]:
# Execute Headless Agent Simulation
print("--- 🤖 RL AGENT HEADLESS VIEW ---")
metrics_df = run_headless_simulation(engine, _inputs)
display(metrics_df.style.format("{:.6f}"))
print(f"\nTarget Reward Signal: {metrics_df.loc['Group Gain', 'Holding']:.6f}")

--- 🤖 RL AGENT HEADLESS VIEW ---
----------------------------------------------------------------------
Timeline: [2025-10-31] -> Decision: 2026-08-05 -> Entry: 2026-08-06 -> End: 2026-08-13
Selected Tickers (50):
SGOV, SHV, BIL, VBIL, MINT, USFR, BOXX, PULS, JPST, JAAA
RVMD, RY, TD, PEN, SNDK, SCHD, MU, JAZZ, IWD, ROST
NUE, HST, DELL, FNDX, HSBC, ASX, RPRX, EWT, STT, VTRS
BNS, STM, VTV, CSCO, TRGP, STX, ATI, HPE, WDC, CASY
BUD, EFV, LITE, FTNT, SN, AMAT, TRV, SCHV, SU, CSX
----------------------------------------------------------------------


,Full,Lookback,Holding
Metric,,,
Group Gain,0.640954,0.616065,0.021101
Benchmark Gain,0.139705,0.129251,0.012054
== Gain Delta,0.501248,0.486814,0.009047
Group Sharpe,3.325073,3.276186,10.848131
Benchmark Sharpe,1.410334,1.334308,8.973867
== Sharpe Delta,1.914739,1.941878,1.874263
Group Sharpe (ATRP),0.104262,0.104903,0.136542
Benchmark Sharpe (ATRP),0.064430,0.061590,0.225417
== Sharpe (ATRP) Delta,0.039832,0.043312,-0.088875



Target Reward Signal: 0.021101


 # Section 5: Stage 2 - Cascade Filter Execution

In [ ]:
print("--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---")
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# LAUNCH STAGE 2 (Cascade on the Stage 1 subset)
analyzer2, stage2_pack = create_walk_forward_analyzer(
    engine,
    universe_subset=analyzer1.last_run.tickers,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

--- CASCADE FILTER, TICKERS PASSED FIRST FILTER ---
🚀 Ready for Stage 2: Run Simulation for 2nd filter.


In [14]:
# Parse and map results from Analyzer 2
result = SU.map_analyzer(analyzer2)

[SEARCH] ANALYZER SIMULATION MAP
[  0] [DATA] audit_pack (EngineOutput)
[  1]   [PLOT] portfolio_series (shape=(196,))
[  2]   [PLOT] benchmark_series (shape=(196,))
[  3]   [CALC] normalized_plot_data (shape=(196, 50))
[  4]   [FILE] tickers (len=50)
[  5]     [DOC] index_0 (str)
[  6]     [DOC] index_1 (str)
[  7]     [DOC] index_2 (str)
[  8]     [DOC] index_3 (str)
[  9]     [DOC] index_4 (str)
[ 10]     [DOC] index_5 (str)
[ 11]     [DOC] index_6 (str)
[ 12]     [DOC] index_7 (str)
[ 13]     [DOC] index_8 (str)
[ 14]     [DOC] index_9 (str)
[ 15]     [DOC] index_10 (str)
[ 16]     [DOC] index_11 (str)
[ 17]     [DOC] index_12 (str)
[ 18]     [DOC] index_13 (str)
[ 19]     [DOC] index_14 (str)
[ 20]     [DOC] index_15 (str)
[ 21]     [DOC] index_16 (str)
[ 22]     [DOC] index_17 (str)
[ 23]     [DOC] index_18 (str)
[ 24]     [DOC] index_19 (str)
[ 25]     [DOC] index_20 (str)
[ 26]     [DOC] index_21 (str)
[ 27]     [DOC] index_22 (str)
[ 28]     [DOC] index_23 (str)
[ 29]     [DOC

In [15]:
# Test different methods to extract resulting target tickers
_tickers = SU.fetch(4, result)
print(f"_tickers (by index): {_tickers}\n")

_tickers = SU.fetch("tickers", result)
print(f"_tickers (by key): {_tickers}\n")

print(f'result[4]["obj"]: {result[4]["obj"]}')

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'HSBC',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'BNS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV',
 'SCHV',
 'SU',
 'CSX']

_tickers (by index): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']

 [INFO] INDEX: [4]
 [LABEL] NAME:  tickers
 [FILE] PATH:  audit_pack -> tickers



['SGOV',
 'SHV',
 'BIL',
 'VBIL',
 'MINT',
 'USFR',
 'BOXX',
 'PULS',
 'JPST',
 'JAAA',
 'RVMD',
 'RY',
 'TD',
 'PEN',
 'SNDK',
 'SCHD',
 'MU',
 'JAZZ',
 'IWD',
 'ROST',
 'NUE',
 'HST',
 'DELL',
 'FNDX',
 'HSBC',
 'ASX',
 'RPRX',
 'EWT',
 'STT',
 'VTRS',
 'BNS',
 'STM',
 'VTV',
 'CSCO',
 'TRGP',
 'STX',
 'ATI',
 'HPE',
 'WDC',
 'CASY',
 'BUD',
 'EFV',
 'LITE',
 'FTNT',
 'SN',
 'AMAT',
 'TRV',
 'SCHV',
 'SU',
 'CSX']

_tickers (by key): ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']

result[4]["obj"]: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']


 # Section 6: Verification, Exports & Auditing

In [16]:
# Export analytical trace logs to Excel
SU.export_audit_to_excel(
    audit_pack=analyzer2.last_run, filename="Audit_Verification_Report.xlsx"
)

[FILE] [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\Audit_Verification_Report.xlsx
[DONE] Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v3/output/Audit_Verification_Report.xlsx')

In [17]:
# Export last-run stacked OHLCV ticker records
SU.export_last_run_tickers_data_to_csv(
    analyzer=analyzer2,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename="last_run_tickers_stacked.csv",
)

Creating combined dictionary for 51 ticker(s)
Date range: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Data retrieved for 51 ticker(s) from 2025-10-31 00:00:00 to 2026-08-13 00:00:00
Total rows: 9996
Date range in data: 2025-10-31 00:00:00 to 2026-08-13 00:00:00
  SGOV: 196 rows
  SHV: 196 rows
  BIL: 196 rows
  VBIL: 196 rows
  MINT: 196 rows
  USFR: 196 rows
  BOXX: 196 rows
  PULS: 196 rows
  JPST: 196 rows
  JAAA: 196 rows
  RVMD: 196 rows
  RY: 196 rows
  TD: 196 rows
  PEN: 196 rows
  SNDK: 196 rows
  SCHD: 196 rows
  MU: 196 rows
  JAZZ: 196 rows
  IWD: 196 rows
  ROST: 196 rows
  NUE: 196 rows
  HST: 196 rows
  DELL: 196 rows
  FNDX: 196 rows
  HSBC: 196 rows
  ASX: 196 rows
  RPRX: 196 rows
  EWT: 196 rows
  STT: 196 rows
  VTRS: 196 rows
  BNS: 196 rows
  STM: 196 rows
  VTV: 196 rows
  CSCO: 196 rows
  TRGP: 196 rows
  STX: 196 rows
  ATI: 196 rows
  HPE: 196 rows
  WDC: 196 rows
  CASY: 196 rows
  BUD: 196 rows
  EFV: 196 rows
  LITE: 196 rows
  FTNT: 196 rows
  SN: 196 rows


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v3/output/last_run_tickers_stacked.csv')

 # Section 7: Slicing Tests & Dictionary Parsing

In [18]:
# Verify captured ticker names
print(f"_tickers: {_tickers}")

_tickers: ['SGOV', 'SHV', 'BIL', 'VBIL', 'MINT', 'USFR', 'BOXX', 'PULS', 'JPST', 'JAAA', 'RVMD', 'RY', 'TD', 'PEN', 'SNDK', 'SCHD', 'MU', 'JAZZ', 'IWD', 'ROST', 'NUE', 'HST', 'DELL', 'FNDX', 'HSBC', 'ASX', 'RPRX', 'EWT', 'STT', 'VTRS', 'BNS', 'STM', 'VTV', 'CSCO', 'TRGP', 'STX', 'ATI', 'HPE', 'WDC', 'CASY', 'BUD', 'EFV', 'LITE', 'FTNT', 'SN', 'AMAT', 'TRV', 'SCHV', 'SU', 'CSX']


In [19]:
# Test various pandas multi-index slicing configurations
##############
test_tickers = ["NVDA", "GOOG"]
##############

idx = pd.IndexSlice
columns = ["Adj Close", "Volume"]

# # Slice 1: All dates, all columns
# display(df_ohlcv.loc[idx[test_tickers]].copy().head(3))

# # Slice 2: specific start date, all columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], :].head(3))

# # Slice 3: specific start date, selected columns
# display(df_ohlcv.loc[idx[test_tickers, pd.Timestamp("2026-04-01") :], columns].head(3))

# Slice 4: specific start/end range, all columns
display(
    df_ohlcv.loc[
        idx[test_tickers, pd.Timestamp("2026-08-07") : pd.Timestamp("2026-08-14")], :
    ]
)

Adj Open  Adj High  Adj Low  Adj Close       Volume
Ticker Date                                                           
NVDA   2026-08-07   221.292   224.509  220.413    223.710  105787639.0
       2026-08-10   223.150   223.889  216.528    217.307  115976231.0
       2026-08-11   221.922   221.952  215.958    217.257  101386421.0
       2026-08-12   220.793   224.848  219.954    223.840  108905327.0
       2026-08-13   224.808   226.976  223.460    225.048   98977826.0
       2026-08-14   226.517   227.236  224.249    224.908   75765583.0
GOOG   2026-08-07   355.049   357.318  352.591    353.241   14526124.0
       2026-08-10   354.010   355.699  351.242    355.609   12797803.0
       2026-08-11   353.770   354.770  342.348    342.777   18402139.0
       2026-08-12   345.106   345.276  339.600    342.148   15618933.0
       2026-08-13   344.346   345.606  342.178    343.717   15007536.0
       2026-08-14   344.037   347.834  341.848    343.317   10953907.0

In [20]:
# Test custom combined dictionary extraction
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=_tickers,
    date_start="2026-05-01",
    date_end=None,
    verbose=False,
)

{'SGOV':             Adj Open  Adj High   Adj Low  Adj Close      Volume       ATR      ATRP       TRP       RSI    Mom_21    Mom_63   Mom_126  Mom_252_21  Consistency     IR_63   Beta_63   IVol_63  SemiDev_63  Trend_R2_63  DD_21    Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount  RecentStaleDays  IsZeroPrice
 Date                                                                                                                                                                                                                                                                                                                        
 2026-05-01   99.2097   99.2097   99.1998    99.2097  34218740.0  0.019255  0.000194  0.000378  0.999997  0.003168  0.009115  0.018944    0.037157          0.6 -0.061536 -0.003311  0.000116         0.0     0.999421    0.0  0.000378         0.011905      1.368668e+09                  0.0              0.0            0
 2026-05-04   99.2196   99.2196   99.2

 # Section 8: Finviz Ingestion & Filtering

In [21]:
# Load latest updated Finviz dataset dynamically
def load_latest_finviz_parquet(data_dir: Path) -> pd.DataFrame:
    pattern = "[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]_df_finviz_merged_stocks_etfs.parquet"
    files = list(data_dir.glob(pattern))

    if not files:
        print(f"DEBUG: No files matching pattern found in {data_dir.absolute()}")
        return None

    latest_path = max(files, key=lambda x: x.name)
    print(f"DEBUG: Loading file: {latest_path.name}")

    try:
        return pd.read_parquet(latest_path)
    except Exception as e:
        print(f"DEBUG: Failed to read {latest_path.name}. Error: {e}")
        return None


# Load the file using the unified data_dir path
df_finviz = load_latest_finviz_parquet(GLOBAL_DATA_DIR)
print(
    f"df_finviz raw records shape: {df_finviz.shape if df_finviz is not None else None}"
)

DEBUG: Loading file: 2026-09-21_df_finviz_merged_stocks_etfs.parquet
df_finviz raw records shape: (1424, 139)


In [22]:
display(df_finviz.head())
display(df_finviz.tail())

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NVDA,1,NVIDIA Corp,"DJIA, NDX, S&P 500",Technology,Semiconductors,USA,NASD,"Technology, Semiconductors",5484440.0,1,5484440.0,28.77,NaN,0.22,18.10,24.00,55.19,43.18,9.48,4.12,0.33,0.52,9/10/2026,0.82,7.91,2.47,NaN,NaN,NaN,NaN,NaN,105.85,128.21,124.42,83.38,302970.0,192880.0,6.22,4.28,24150.0,23190.0,96.02,3.79,-0.55,72.33,0.50,1.29,2.31,298.30,83.68,117.21,72.42,4.59,3.85,0.16,0.17,74.67,65.21,63.66,4.760334,7.87,4.94,9.07,31.77,28.81,22.02,2.22,6.46,2.838687,1.72,2.55,3.67,6.04,14.58,-3.06,19.77,-3.79,38.53,164.27 - 236.54,-3.79,682609.99,58.37,Aug 26/a,1/22/1999,Yes,Yes,42000.0,2.08,0.30,1.21,129.28,0.67,73945949,333.47,222.27,222.93,228.50,221.56,227.57,2.38,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,35.767295,83978.173388,7482.370304,23.546841,1745.384500,220.897778,-3.181378,-3.934942,0.601205,1.335837,1.981019,1.224065,-0.039142,-0.064253,0.993169,1.467807,2.367773,1.266458,1.672178,2.662954,1.310487,0.599051,0.887754,1.101187
AAPL,2,Apple Inc,"DJIA, NDX, S&P 500",Technology,Consumer Electronics,USA,NASD,"Technology, Consumer Electronics",4939250.0,2,4939250.0,38.80,NaN,2.76,10.58,45.98,79.16,36.14,7.36,4.28,0.33,1.06,8/10/2026,13.66,8.72,1.98,NaN,NaN,NaN,NaN,NaN,16.36,29.13,32.57,14.24,466820.0,128930.0,6.77,0.35,14610.0,14580.0,99.78,0.12,-2.27,68.92,0.23,0.96,2.61,139.75,36.08,148.75,72.08,1.00,0.93,0.66,0.78,48.65,33.17,27.62,1.445645,1.61,8.72,13.95,36.47,37.86,24.49,1.07,7.23,2.136272,1.60,2.23,4.43,5.56,18.09,-1.78,12.81,-1.78,40.89,240.21 - 344.57,-1.78,531904.47,65.95,Jul 30/a,12/12/1980,Yes,Yes,166000.0,0.90,-0.21,2.13,53.44,0.45,20376237,337.68,336.13,335.41,339.26,333.05,338.44,0.69,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,7.471411,44.686296,4.980972,4.042080,11.297335,2.058357,5.942827,18.923899,3.127207,3.551583,6.986322,1.855747,2.764824,4.942721,1.575505,2.925687,4.522104,1.680181,2.352870,3.482002,1.501572,1.095498,1.585704,1.214814
GOOG,3,Alphabet Inc,"NDX, S&P 500",Communication Services,Internet Content & Information,USA,NASD,"Communication Services, Internet Content & Inf...",4330120.0,3,4330120.0,17.64,NaN,1.26,9.70,6.90,17.86,81.28,50.90,19.67,0.23,0.86,9/4/2026,7.68,19.91,3.03,NaN,NaN,NaN,NaN,NaN,23.98,294.02,112.71,20.23,446310.0,244120.0,216.66,2.96,5530.0,5130.0,92.85,58.04,-0.01,28.68,2.31,0.88,2.29,45.27,34.30,48.68,32.33,2.72,2.64,0.18,0.18,60.94,33.96,54.70,0.872800,1.59,3.84,0.69,17.54,37.59,11.92,1.20,8.10,2.306444,2.17,1.84,3.62,2.27,4.41,-8.02,11.53,-13.17,48.38,236.68 - 404.47,-13.17,1344.56,58.73,Jul 22/a,3/27/2014,Yes,Yes,190820.0,1.47,

,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
NEAR,394,iShares Short Duration Bond Active ETF,-,Financial,Exchange Traded Fund,USA,CBOE,"Financial, Exchange Traded Fund, Bonds - Broad...",5170.0,1554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.46,2.24,9/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.49,0.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.119713,-0.04,-0.91,-0.87,-1.14,-2.30,-1.97,0.03,0.07,0.139748,0.12,0.12,-0.43,-0.65,-1.35,-1.10,0.06,-2.49,0.06,50.06 - 51.37,-2.56,13.92,30.70,-,9/26/2013,Yes,Yes,NaN,-0.08,0.14,NaN,0.57712,0.76,376558,NaN,50.06,50.13,50.13,50.08,50.09,0.06,Bonds - Broad Market,Bonds,0.25,1852.0,5170.0,NaN,6.55,NaN,11.73,1500.00,40.75,2.00,5.08,3.85,"U.S., fixed-income, bonds, investment-grade",-2.872565,-4.574482,0.592473,-5.338413,-6.478135,0.398433,-11.643926,-10.159125,0.137046,-9.428645,-8.705151,0.183970,-6.221555,-6.323637,0.326173,-3.527721,-4.028819,0.557127,-1.666700,-2.099365,0.760434,-1.483491,-1.871562,0.783328
ISTB,395,iShares Core 1-5 Year USD Bond ETF,-,Financial,Exchange Traded Fund,USA,NASD,"Financial, Exchange Traded Fund, Bonds - Broad...",5120.0,1556,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.33,2.06,9/1/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.50,0.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.063198,-0.03,-1.13,-1.30,-1.71,-2.81,-2.59,0.11,0.09,0.189514,0.17,0.13,-0.55,-0.89,-1.82,-1.38,0.14,-3.19,0.14,47.42 - 49.05,-7.76,5.33,32.97,-,10/22/2012,Yes,Yes,NaN,-0.05,0.15,NaN,0.40023,1.20,412990,NaN,47.44,47.51,47.51,47.47,47.49,0.09,Bonds - Broad Market,Bonds,0.06,7481.0,5120.0,NaN,2.19,NaN,4.18,557.85,12.22,1.24,4.75,1.75,"U.S., fixed-income, bonds",0.289587,0.594512,1.052963,-4.100113,-5.985647,0.532966,-9.689687,-9.380294,0.194060,-7.874836,-7.875470,0.238868,-5.577904,-6.008576,0.381162,-3.430746,-4.047934,0.581376,-1.744285,-2.213409,0.762340,-1.461171,-1.850515,0.791789
BTC,397,Grayscale Bitcoin Mini Trust ETF,-,Financial,Exchange Traded Fund,USA,NYSE,"Financial, Exchange Traded Fund, Other Asset T...",5100.0,1557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.25,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.747919,9.02,18.38,33.64,22.58,-25.34,-1.73,1.60,1.16,3.047819,2.59,2.42,9.33,22.00,15.85,5.20,39.36,-31.99,48.38,25.65 - 55.96,-31.99,73.00,71.66,-,7/31/2024,Yes,Yes,NaN,1.38,4.54,NaN,1.82000,1.30,2038630,NaN,35.91,37.54,38.18,37.48,38.06,5.99,Other Asset Types - Multi-A

In [23]:
raw_text = """
SMTC, TER, MTSI, PRAX, TTMI, LSCC,
"""

# Split by comma, strip whitespace, and filter out any empty strings
ticker_list = [item.strip() for item in raw_text.split(",") if item.strip()]

print(ticker_list)

['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [24]:
# Set test target tickers
##################
# target_tickers = ["GLW", "NOK", "DELL", "NVDA", "GOOG"]
target_tickers = ticker_list

print(f"target_tickers: {target_tickers}")
##################

target_tickers: ['SMTC', 'TER', 'MTSI', 'PRAX', 'TTMI', 'LSCC']


In [25]:
# Clean index extraction using ast exception parsing if tickers are missing
try:
    result = df_finviz.loc[target_tickers]
except KeyError as e:
    error_msg = str(e)
    missing_str = error_msg.replace(" not in index", "").strip('"').strip("'")
    missing_tickers = ast.literal_eval(missing_str)

    print(f"Missing tickers: {missing_tickers}")

    # Strip out the missing symbols and retry slicing
    target_tickers_in_finviz = [t for t in target_tickers if t not in missing_tickers]
    result = df_finviz.loc[target_tickers_in_finviz].copy()
    result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

print(f"result shape: {result.shape}")

Missing tickers: ['PRAX']
result shape: (5, 139)


In [26]:
# Display filtered Finviz information and schema details
result.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, TER to TTMI
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 5.5+ KB


In [27]:
# Check specific target column (Market Capitalization)
result["MktCap AUM, M"]

TER     59600.0
MTSI    21250.0
LSCC    16670.0
SMTC    16610.0
TTMI    13210.0
Name: MktCap AUM, M, dtype: float64

In [28]:
# Sort records descendingly by Market Capitalization
result.sort_values(by="MktCap AUM, M", ascending=False, inplace=True)

In [29]:
# View final sorted DataFrame
print(result)

      No.                                  Company         Index      Sector                             Industry Country Exchange                                             Info  MktCap AUM, M  Rank  Market Cap, M     P/E  Fwd P/E   PEG    P/S    P/B     P/C   P/FCF  Book/sh  Cash/sh  Dividend %  Dividend TTM Dividend Ex Date  Payout Ratio %   EPS  EPS next Q  EPS this Y %  EPS next Y %  EPS past 5Y %  EPS next 5Y %  Sales past 5Y %  Sales Q/Q %  EPS Q/Q %  EPS YoY TTM %  Sales YoY TTM %  Sales, M  Income, M  EPS Surprise %  Revenue Surprise %  Outstanding, M  Float, M  Float %  Insider Own %  Insider Trans %  Inst Own %  Inst Trans %  Short Float %  Short Ratio  Short Interest, M  ROA %  ROE %  ROIC %  Curr R  Quick R  LTDebt/Eq  Debt/Eq  Gross M %  Oper M %  Profit M %  Perf 3D %  Perf Week %  Perf Month %  Perf Quart %  Perf Half %  Perf Year %  Perf YTD %  Beta    ATR  ATR/Price %  Volatility W %  Volatility M %  SMA20 %  SMA50 %  SMA200 %  50D High %  50D Low %  52W High %  52W 

In [30]:
# # Export processed dataframe output directly to trash file
# result.to_csv(output_dir / "_trash.csv")